In [1]:
print("hi")

hi


In [3]:
!pip install roboflow

from roboflow import Roboflow

# Initialize roboflow with your API key
rf = Roboflow(api_key="56ROJMsyNuusXttM8RAf")
project = rf.workspace("coursework-y7x7o").project("eye-disease-classification-iu3lf")
version = project.version(2)

# Download the dataset in folder format (this creates proper train/valid/test folders with class subfolders)
dataset = version.download("folder")

print(f"Dataset downloaded to: {dataset.location}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 69.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Eye-disease-classification-2 in folder:: 100%|██████████| 1196/1196 [00:00<00:00, 4873.26it/s]

Dataset downloaded to: /content/Eye-disease-classification-2


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader, random_split
import os

# -------------------------------
# Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------
# Transforms
# -------------------------------
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

transform_eval = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# -------------------------------
# Load Dataset
# -------------------------------
data_path = "/content/Eye-disease-classification-2"

full_train = datasets.ImageFolder(os.path.join(data_path, "train"), transform=transform_train)
test_dataset = datasets.ImageFolder(os.path.join(data_path, "test"), transform=transform_eval)

# Custom train/val split (80/20)
val_size = int(0.2 * len(full_train))
train_size = len(full_train) - val_size
train_dataset, valid_dataset = random_split(full_train, [train_size, val_size])

# Apply eval transform to valid dataset
valid_dataset.dataset.transform = transform_eval

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

print("Classes:", full_train.classes)

# -------------------------------
# Define Model (ResNet18 + Dropout + LabelSmoothing)
# -------------------------------
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, len(full_train.classes))
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # 🔑 label smoothing

# -------------------------------
# Stage 1 - Train only FC
# -------------------------------
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)

print("🔹 Stage 1 Training...")
for epoch in range(5):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, pred = outputs.max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
    train_acc = 100*correct/total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, pred = outputs.max(1)
            val_correct += pred.eq(labels).sum().item()
            val_total += labels.size(0)
    val_acc = 100*val_correct/val_total

    print(f"[Stage1] Epoch {epoch+1}/5, "
          f"Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, "
          f"Val Acc: {val_acc:.2f}%")

# -------------------------------
# Stage 2 - Fine-tune ALL layers with OneCycleLR + Early Stopping
# -------------------------------
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-4, steps_per_epoch=len(train_loader), epochs=15
)

best_val_acc = 0
patience, patience_counter = 3, 0

print("🔹 Stage 2 Fine-tuning...")
for epoch in range(15):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        _, pred = outputs.max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
    train_acc = 100*correct/total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, pred = outputs.max(1)
            val_correct += pred.eq(labels).sum().item()
            val_total += labels.size(0)
    val_acc = 100*val_correct/val_total

    print(f"[Stage2] Epoch {epoch+1}/15, "
          f"Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, "
          f"Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("⏹ Early stopping triggered!")
            break

# -------------------------------
# Final Test
# -------------------------------
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, pred = outputs.max(1)
        test_correct += pred.eq(labels).sum().item()
        test_total += labels.size(0)
test_acc = 100*test_correct/test_total
print(f"✅ Final Test Accuracy: {test_acc:.2f}%")


Using device: cpu
Classes: ['Cataracts', 'Normal_Eyes', 'Uveitis']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 95.1MB/s]


🔹 Stage 1 Training...
[Stage1] Epoch 1/5, Loss: 1.2524, Train Acc: 38.05%, Val Acc: 54.69%
[Stage1] Epoch 2/5, Loss: 1.0048, Train Acc: 53.64%, Val Acc: 69.79%
[Stage1] Epoch 3/5, Loss: 0.9126, Train Acc: 62.86%, Val Acc: 75.52%
[Stage1] Epoch 4/5, Loss: 0.8710, Train Acc: 64.55%, Val Acc: 75.52%
[Stage1] Epoch 5/5, Loss: 0.8257, Train Acc: 67.79%, Val Acc: 70.83%
🔹 Stage 2 Fine-tuning...
[Stage2] Epoch 1/15, Loss: 0.8150, Train Acc: 69.09%, Val Acc: 72.92%
[Stage2] Epoch 2/15, Loss: 0.6927, Train Acc: 77.01%, Val Acc: 81.77%
[Stage2] Epoch 3/15, Loss: 0.4925, Train Acc: 88.83%, Val Acc: 86.98%
[Stage2] Epoch 4/15, Loss: 0.4122, Train Acc: 96.62%, Val Acc: 84.90%
[Stage2] Epoch 5/15, Loss: 0.3830, Train Acc: 98.57%, Val Acc: 86.98%
[Stage2] Epoch 6/15, Loss: 0.3828, Train Acc: 98.57%, Val Acc: 88.54%
[Stage2] Epoch 7/15, Loss: 0.3668, Train Acc: 99.09%, Val Acc: 89.06%
[Stage2] Epoch 8/15, Loss: 0.3699, Train Acc: 99.22%, Val Acc: 89.58%
[Stage2] Epoch 9/15, Loss: 0.3525, Train Acc: 99

In [10]:
# After finishing training (Stage 2)
torch.save(model.state_dict(), "best_model.pth")
print("✅ Model saved as best_model.pth")


✅ Model saved as best_model.pth


In [11]:
from google.colab import files

files.download("best_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
torch.save(model.state_dict(), "eye_disease_model.pth")
